# 🎭 페르소나 파인튜닝 — Unsloth (Colab 무료, 텍스트 모델)

> 내 디지털 분신 팀을 LoRA로 학습. 맨 위 `AGENT`만 바꾸면 5명 모두 가능.

**런타임 → 런타임 유형 변경 → T4 GPU** 먼저!  그리고 **위에서부터 순서대로 ▶**.

> 🆕 개선판: ①한국어+페르소나 고정 **시스템 프롬프트**, ②데이터 보강(53개), ③**덜 외우게**(에폭 ↓) → 실제 대화에서 횡설수설/외국어 섞임 방지.

## ① 설정

In [ ]:
AGENT = "cora"   # cora | finn | offie | rina | geo
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LEN = 1024
DATA_FILE = f"{AGENT}.jsonl"

# 페르소나를 한국어로 고정하는 시스템 프롬프트 (외국어 새는 것 방지)
SYSTEMS = {
    "cora":  "너는 '코라'야. 한국어로만 말하는 시적 글쓰기 도우미지. 항상 차분한 반말과 비유로 따뜻하고 짧게 말해.",
    "finn":  "너는 '핀'이야. 한국어로만 말하는 AI 수익화 도우미지. 차분한 반말과 강물·물길 비유로 현실적으로 말해.",
    "offie": "너는 '오피'야. 한국어로만 말하는 1인 사업 운영 파트너지. 차분한 반말로 군더더기 없이 말해.",
    "rina":  "너는 '리나'야. 한국어로만 말하는 모임·커뮤니티 도우미지. 따뜻한 반말과 별·온기 비유로 말해.",
    "geo":   "너는 '지오'야. 한국어로만 말하는 사용자의 디지털 분신이자 길잡이지. 차분한 반말로 큰 그림을 잡아주고, 필요하면 코라·핀·오피·리나에게 연결해.",
}
SYSTEM = SYSTEMS[AGENT]
print(f"▶ {AGENT} | {MODEL_NAME}\n시스템: {SYSTEM}")

## ② 설치

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo

## ③ 모델 로드

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)

## ④ 학습 전 베이스라인 + 대화 함수

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def ask(model, question, history=None, max_new_tokens=200):
    FastLanguageModel.for_inference(model)
    messages = [{"role": "system", "content": SYSTEM}]
    if history:
        messages += history
    messages.append({"role": "user", "content": question})
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.9, repetition_penalty=1.1,
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

print("[학습 전]", ask(model, "넌 누구야?"))

## ⑤ 데이터 업로드 & 변환 (👈 jsonl 올리기)
시스템 프롬프트를 각 대화 앞에 붙여 학습한다(추론과 형식 일치).

In [ ]:
import os
if not os.path.exists(DATA_FILE):
    from google.colab import files
    print(f"⬆️ {DATA_FILE} 업로드")
    files.upload()

from datasets import load_dataset
from unsloth.chat_templates import standardize_data_formats

dataset = load_dataset("json", data_files=DATA_FILE, split="train")
dataset = standardize_data_formats(dataset)
print(f"✅ {len(dataset)}개 로드")

def formatting(examples):
    texts = []
    for convo in examples["conversations"]:
        msgs = [{"role": "system", "content": SYSTEM}] + list(convo)
        texts.append(tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

dataset = dataset.map(formatting, batched=True)
print(dataset[0]["text"][:500])

## ⑥ LoRA 부착

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## ⑦ 학습 설정
🎯 이번엔 **에폭 수**로 조절(덜 외우게). `num_train_epochs = 4`가 기본.
- 학습 후 너무 **딱딱하게 데이터만 반복**하면 → 3으로 ↓
- 말투가 **약하면** → 5~6으로 ↑
- 목표 Loss **0.3~0.6** (0.05처럼 너무 낮으면 과적합!)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 4,          # ← 과적합 방지 (max_steps 대신 에폭)
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

## ⑧ 학습 🔥 (Loss 0.3~0.6 목표)

In [ ]:
trainer_stats = trainer.train()

## ⑨ 테스트 🎉

In [ ]:
tests = {
    "cora":  ["넌 누구야?", "첫 문장이 안 써져", "안녕!", "오늘 기분이 안 좋아"],
    "finn":  ["넌 누구야?", "AI로 돈 벌기 뭐부터 시작해?", "안녕!"],
    "offie": ["넌 누구야?", "일이 너무 많아서 숨이 막혀", "안녕!"],
    "rina":  ["넌 누구야?", "사람이 잘 안 모여서 속상해", "안녕!"],
    "geo":   ["넌 누구야?", "하고 싶은 게 너무 많아", "안녕!"],
}
for q in tests.get(AGENT, ["넌 누구야?"]):
    print(f"Q: {q}\nA: {ask(model, q)}\n")

## ⑩ 직접 대화해보기 💬 (끝내려면 quit)

In [ ]:
history = []
print(f"🎭 {AGENT}와 대화! (quit 입력 시 종료)\n")
while True:
    user = input("나: ")
    if user.strip().lower() in ("quit", "exit", "그만"):
        print("또 보자!"); break
    reply = ask(model, user, history=history)
    print(f"{AGENT}:", reply, "\n")
    history += [{"role": "user", "content": user},
                {"role": "assistant", "content": reply}]
    history = history[-8:]  # 최근 4턴만 기억

## ⑪ 저장 💾

In [ ]:
save_dir = f"{AGENT}_lora"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
import shutil
shutil.make_archive(save_dir, "zip", save_dir)
from google.colab import files
files.download(f"{save_dir}.zip")
print(f"💾 {save_dir}.zip")